# Ingenierie de Prompts pour la Generation d Images avec Stable Diffusion

## Contexte et Objectifs

Ce notebook est un guide complet sur l'art de l'ingenierie de prompts pour `Stable Diffusion`, l'un des modeles de generation d'images les plus puissants. La qualite de l'image generee depend de maniere critique de la qualite et de la precision du prompt. Ce guide vous montrera comment construire des prompts efficaces pour obtenir les resultats souhaites.

### Structure du Notebook :

1.  **Introduction a Stable Diffusion :** Un bref rappel du fonctionnement des modeles de diffusion.
2.  **Configuration et Chargement du Modele :** Nous utilisons un modele Stable Diffusion robuste et bien connu, avec des instructions claires pour le chargement.
3.  **L'Anatomie d'un Prompt Efficace :** Nous decomposons un prompt en ses elements constitutifs :
    *   **Sujet :** L'element principal de l'image.
    *   **Style :** Le style artistique (photorealisme, impressionnisme, etc.).
    *   **Artiste :** Pour imiter le style d'un artiste specifique.
    *   **Medium :** Peinture a l'huile, crayon, etc.
    *   **Details et Modificateurs :** Pour affiner l'eclairage, la composition, le niveau de detail, etc.
4.  **Le Role des Prompts Negatifs :** Comment utiliser les prompts negatifs pour exclure des elements indesirables.
5.  **Prompting Iteratif :** Une demonstration pratique ou nous partons d'un prompt simple et l'enrichissons etape par etape pour ameliorer l'image.
6.  **Techniques Avancees :** Ponderation des mots-cles pour donner plus ou moins d'importance a certains concepts.

_Derniere mise a jour : 2026-02-16_

In [1]:
# --- 1. Installation des Dependances ---
# Note : L'installation de ces bibliotheques peut prendre plusieurs minutes.
%pip install -q diffusers transformers accelerate torch
print("Dependances installees.")

FULL= False
model_id= hf-internal-testing/tiny-stable-diffusion-pipe


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

An error occurred while trying to fetch C:\Users\Olivier Robert\.cache\huggingface\hub\models--hf-internal-testing--tiny-stable-diffusion-pipe\snapshots\3ee6c9f225f088ad5d35b624b6514b091e6a4849\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\Olivier Robert\.cache\huggingface\hub\models--hf-internal-testing--tiny-stable-diffusion-pipe\snapshots\3ee6c9f225f088ad5d35b624b6514b091e6a4849\vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch C:\Users\Olivier Robert\.cache\huggingface\hub\models--hf-internal-testing--tiny-stable-diffusion-pipe\snapshots\3ee6c9f225f088ad5d35b624b6514b091e6a4849\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\Olivier Robert\.cache\huggingface\hub\models--hf-internal-testing--tiny-stable-diffusion-pipe\snapshots\3ee6c9f225f088ad5d35b624b6514b091e6a4849\unet.
Defaulting to unsafe serialization. Pass `a

  0%|          | 0/5 [00:00<?, ?it/s]

RuntimeError: The size of tensor a (12545) must match the size of tensor b (226) at non-singleton dimension 1

In [2]:
# --- 2. Imports et Configuration ---
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import matplotlib.pyplot as plt
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

NameError: name 'image' is not defined

<Figure size 500x500 with 0 Axes>

## 3. Chargement du Modele Stable Diffusion

Nous allons utiliser le modele `runwayml/stable-diffusion-v1-5`, une version populaire et robuste. Le chargement peut prendre du temps car le modele est volumineux. Pour les utilisateurs avec des contraintes materielles, un modele plus petit (`CompVis/stable-diffusion-v1-4`) peut etre utilise, mais avec une qualite potentiellement inferieure.

In [3]:
# Modele Stable Diffusion v1.5
model_id = "runwayml/stable-diffusion-v1-5"

# Utiliser le GPU si disponible (CUDA), sinon le CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Utilisation du device : {device}")

# Charger le pipeline. Utiliser float16 pour economiser de la memoire sur GPU
torch_dtype = torch.float16 if device == "cuda" else torch.float32

try:
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch_dtype)
    pipe = pipe.to(device)
    logger.info("Pipeline Stable Diffusion charge avec succes.")
except Exception as e:
    logger.error(f"Erreur lors du chargement du modele : {e}. Cela peut etre du a des contraintes de memoire.")
    # En cas d'echec, on ne definit pas le pipe pour arreter l'execution.
    pipe = None

DONE 2026-02-16 07:20:44


## 4. L'Anatomie d'un Prompt

Un prompt efficace combine plusieurs elements pour guider le modele avec precision.

`(Sujet)` + `(Medium)` + `(Style)` + `(Artiste)` + `(Details)` + `(Couleur/Eclairage)`

In [ ]:
# --- Exemple de Prompt Decompose ---

sujet = "A majestic lion"
medium = "oil painting"
style = "impressionistic"
artiste = "in the style of Claude Monet"
details = "highly detailed, on a canvas texture"
eclairage = "soft, golden hour light"

# Combiner les elements pour former le prompt final
prompt_complet = f"{sujet}, {medium}, {style}, {artiste}, {details}, {eclairage}"

print("Prompt Complet :")
print(prompt_complet)

# Prompt negatif pour eviter certains defauts courants
prompt_negatif = "blurry, low quality, deformed, ugly, watermark"

# Fonction pour generer et afficher une image
def generer_et_afficher(prompt, negative_prompt, num_steps=25):
    if pipe is None:
        logger.warning("Pipeline non initialise. Generation d'image ignoree.")
        return
    
    logger.info(f"Generation d'image avec {num_steps} etapes...")
    # Definir une seed pour la reproductibilite
    generator = torch.Generator(device=device).manual_seed(42)
    
    with torch.autocast(device):
        image = pipe(
            prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=num_steps,
            generator=generator
        ).images[0]
    
    logger.info("Image generee avec succes.")
    
    # Affichage
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Prompt: {prompt[:80]}...")
    plt.show()

# Generer l'image avec notre prompt detaille
generer_et_afficher(prompt_complet, prompt_negatif)

## 5. Prompting Iteratif : Ameliorer une Idee Simple

Voyons comment un prompt simple peut etre progressivement ameliore.

In [ ]:
# --- Iteration 1 : Prompt de base ---
prompt_1 = "A futuristic car"
generer_et_afficher(prompt_1, prompt_negatif, num_steps=20)

# --- Iteration 2 : Ajout de style et de details ---
prompt_2 = "A futuristic cyberpunk car, neon lights, driving through a rainy city at night"
generer_et_afficher(prompt_2, prompt_negatif, num_steps=20)

# --- Iteration 3 : Ajout de qualite et d'eclairage cinematographique ---
prompt_3 = "A futuristic cyberpunk car, stunningly beautiful, cinematic lighting, neon lights reflecting on wet streets, high detail, 8k"
generer_et_afficher(prompt_3, prompt_negatif, num_steps=25)

## 6. Technique Avancee : Ponderation des Mots-cles

On peut utiliser des parentheses `()` pour augmenter l'importance d'un mot et des crochets `[]` pour la diminuer. La syntaxe `(mot:poids)` permet un controle encore plus fin (poids > 1 pour augmenter, < 1 pour diminuer).

In [ ]:
# --- Exemple de Ponderation ---

# Sans ponderation, le modele pourrait melanger les deux concepts
prompt_sans_poids = "A cat wearing a hat"
generer_et_afficher(prompt_sans_poids, prompt_negatif, num_steps=20)

# Avec ponderation, on insiste sur le chapeau
prompt_avec_poids = "A cat wearing a (top hat:1.4)"
generer_et_afficher(prompt_avec_poids, prompt_negatif, num_steps=20)

## Conclusion

L'ingenierie de prompts est un processus iteratif et creatif. En combinant un sujet clair, un style, des details et des modificateurs, et en utilisant des prompts negatifs, vous pouvez guider Stable Diffusion pour creer des images qui correspondent precisement a votre vision. La ponderation des mots-cles offre un niveau de controle supplementaire pour affiner encore davantage les resultats.